<a href="https://colab.research.google.com/github/brasilemnumeros/dados/blob/main/indicadores/meio-ambiente/Desmatamento.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Desmatamento da Amazônia Legal (km²)

Este notebook é o motor de coleta e publicação da série `desmatamento-amazonia-legal-anual`.

- **Fonte:** INPE — PRODES/BiomasBR, via TerraBrasilis. A taxa é a **consolidada** da **Amazônia Legal**, calculada pelo site como a soma dos **9 estados**.
- **Página da fonte:** `https://terrabrasilis.dpi.inpe.br/app/dashboard/deforestation/biomes/legal_amazon/rates`
- **Arquivo da fonte:** `rates<ano>.json`, cujo nome gira a cada ano. O notebook descobre o nome no bundle `app.*.js` da página e cai para `rates<ano civil>.json` e `rates<ano civil − 1>.json` se o bundle não responder.
- **Definição:** área de vegetação nativa suprimida por corte raso na Amazônia Legal no **ano Prodes** (1º de agosto a 31 de julho), em km². É fluxo anual, não estoque acumulado.
- **Fórmula:** soma das áreas de todos os `features` do período (`features[].areas[].area`), exatamente como o site publica. O ano do ponto é o `endDate.year` do período.
- **Unidade:** `numero` (`km²`, 0 casas); `eixo: contagem`; `medida: nivel`; granularidade `anual`.
- **Período publicado:** de 1988 até o último ano consolidado (o ano do arquivo − 1); na execução atual, 1988–2025 (38 pontos). O ano do arquivo é descartado porque a fonte mistura a **estimativa** do ano do arquivo com as taxas consolidadas, sem marcador.
- **Arquivo gerado:** `indicadores/meio-ambiente/desmatamento-amazonia-legal-anual.json`, com `versao: 1` e `id: desmatamento-amazonia-legal-anual`.

## Decisão metodológica

A série nacional de desmatamento não existe: os biomas têm inícios e métodos diferentes (Amazônia em 1988, Cerrado em 2001, três biomas só em 2024). Publicar “Brasil” produziria dois ou três pontos, ou uma emenda invisível. Por isso a série é a **Amazônia Legal**, que tem a maior janela contínua.

O número **não** é: ano-calendário, o Brasil inteiro, o DETER (alertas mensais) nem queimada. A estimativa divulgada em outubro é substituída pela taxa consolidada no primeiro semestre seguinte; a série inteira é reescrita pela fonte, sem changelog.

Os dados do TerraBrasilis são publicados sob **CC BY-SA 4.0**; a atribuição ao INPE/TerraBrasilis está na `fonte` da série e em `docs/metodologia.md`.


## Setup e configuração

`DADOS_RAIZ` permite rodar contra uma cópia do repositório; `MEIO_AMBIENTE_SAIDA` permite ensaio sem tocar no arquivo publicado. Todo caminho impresso passa por `relativo()`, então nenhuma saída commitada carrega o diretório da máquina.


In [1]:
from datetime import date
from pathlib import Path
from urllib.parse import urljoin
import json
import math
import os
import re
import time

import requests

RAIZ = Path(os.environ.get("DADOS_RAIZ", Path.cwd())).resolve()
if os.environ.get("DADOS_RAIZ"):
    PASTA = RAIZ / "indicadores" / "meio-ambiente"
    if not PASTA.is_dir():
        raise SystemExit(f"família não encontrada em {PASTA}")
else:
    PASTA = RAIZ
DESTINO = Path(os.environ.get("MEIO_AMBIENTE_SAIDA", PASTA))
ARQUIVO = DESTINO / "desmatamento-amazonia-legal-anual.json"


def relativo(caminho):
    """Caminho exibido no notebook: relativo à raiz, nunca o absoluto da máquina."""
    try:
        return caminho.relative_to(RAIZ)
    except ValueError:
        return caminho.name


PAGINA = "https://terrabrasilis.dpi.inpe.br/app/dashboard/deforestation/biomes/legal_amazon/rates"
BASE_ARQUIVOS = "https://terrabrasilis.dpi.inpe.br/app/prodes/dashboard/deforestation/files/"
URL_FONTE = PAGINA
HOJE = date.today()
LIMITE = None  # definido depois de escolher o arquivo, pelo ano do nome

print(f"Página da fonte: {PAGINA}")
print(f"Ano civil corrente: {HOJE.year}; o corte será o ano do arquivo − 1")
print(f"Arquivo de saída: {relativo(ARQUIVO)}")


Página da fonte: https://terrabrasilis.dpi.inpe.br/app/dashboard/deforestation/biomes/legal_amazon/rates
Ano civil corrente: 2026; o corte será o ano do arquivo − 1
Arquivo de saída: indicadores/meio-ambiente/desmatamento-amazonia-legal-anual.json


## Resolução do arquivo da fonte

O nome `rates<ano>.json` muda todo ano: `rates2025.json` e `rates2024.json` já respondem 404. A página do TerraBrasilis carrega um bundle `app.*.js` que cita o arquivo vigente. O notebook lê esse nome, ordena os candidatos do ano mais recente para o mais antigo e tenta baixar; a lista sempre inclui `rates<ano civil>.json` e `rates<ano civil − 1>.json` como fallback. A fonte oscila, então cada download repete com espera crescente e timeout generoso.


In [2]:
CABECALHOS = {
    "User-Agent": "brasilemnumeros-dados/1.0 (+https://github.com/brasilemnumeros/dados)"
}


def baixar(url, tentativas=3, timeout=90):
    """Baixa com espera crescente; o TerraBrasilis oscila sem aviso."""
    for tentativa in range(tentativas):
        try:
            resposta = requests.get(url, headers=CABECALHOS, timeout=timeout)
            resposta.raise_for_status()
            return resposta
        except requests.RequestException:
            if tentativa == tentativas - 1:
                raise
            time.sleep(2 ** tentativa)


try:
    pagina = baixar(PAGINA, tentativas=4).text
    bundles = re.findall(r'src="([^"]*app\.[^"]+\.js)"', pagina)
    if not bundles:
        raise RuntimeError("nenhum bundle app.*.js na página")
    bundle_url = urljoin(PAGINA, bundles[0])
    bundle = baixar(bundle_url, tentativas=4).text
    nomes_no_bundle = set(re.findall(r"rates\d{4}\.json", bundle))
    print(f"Bundle consultado: {bundle_url}")
except (RuntimeError, requests.RequestException) as erro:
    print(f"aviso: resolução pelo bundle falhou ({erro}); usando fallback rates<ano>.json")
    nomes_no_bundle = set()

candidatos = nomes_no_bundle | {f"rates{HOJE.year}.json", f"rates{HOJE.year - 1}.json"}
candidatos = sorted(
    candidatos,
    key=lambda nome: int(re.search(r"\d{4}", nome).group()),
    reverse=True,
)
print(f"Candidatos, do mais recente para o mais antigo: {candidatos}")

bruto = None
for nome in candidatos:
    url = BASE_ARQUIVOS + nome
    try:
        resposta = baixar(url, tentativas=3)
        bruto = resposta.json()
    except (requests.RequestException, ValueError) as erro:
        print(f"descartado: {url} ({erro})")
        continue
    print(f"Arquivo da fonte escolhido: {url}")
    break

if bruto is None:
    raise RuntimeError(f"nenhum rates<ano>.json respondeu: {candidatos}")

ANO_ARQUIVO = int(re.search(r"\d{4}", nome).group())
LIMITE = ANO_ARQUIVO - 1
print(f"Ano do arquivo: {ANO_ARQUIVO}; corte em {LIMITE}")


Bundle consultado: https://terrabrasilis.dpi.inpe.br/app/dashboard/deforestation/app.62774204d9672458618b.js
Candidatos, do mais recente para o mais antigo: ['rates2026.json', 'rates2025.json']
Arquivo da fonte escolhido: https://terrabrasilis.dpi.inpe.br/app/prodes/dashboard/deforestation/files/rates2026.json
Ano do arquivo: 2026; corte em 2025


## Cálculo e corte do período fechado

Cada período da fonte tem `startDate`, `endDate` e uma lista de `features`, cada uma com `areas[].area` em km². A soma de todas as áreas reproduz a tabela oficial. O ponto é rotulado pelo ano do `endDate` (o ano Prodes) e recebe a data C15 `AAAA-01-01`.

O corte é `ano do arquivo − 1`: o último período do arquivo é a **estimativa** do ano do arquivo, mesmo que venha sem marcador. O notebook nunca publica esse último período.


In [3]:
periodos = bruto.get("periods")
if not isinstance(periodos, list) or not periodos:
    raise ValueError("a fonte não trouxe a lista periods")

registros = {}
for periodo in periodos:
    inicio = periodo["startDate"]
    fim = periodo["endDate"]
    if (inicio["month"], inicio["day"]) != (8, 1) or (fim["month"], fim["day"]) != (7, 31):
        raise ValueError(f"ano Prodes fora do padrão 1º/08–31/07: {inicio} a {fim}")

    ano = int(fim["year"])
    features = periodo["features"]
    if len(features) != 9:
        raise ValueError(f"período {ano} tem {len(features)} estados, esperado 9")
    if len({feature["loiname"] for feature in features}) != 9:
        raise ValueError(f"período {ano} não tem 9 loinames distintos")

    areas = [
        area["area"]
        for feature in features
        for area in feature["areas"]
    ]
    if not areas:
        raise ValueError(f"período {ano} sem áreas na fonte")

    total = sum(areas)
    if float(total) != int(total) or total <= 0:
        raise ValueError(f"área inválida em {ano}: {total}")
    if ano in registros:
        raise ValueError(f"ano Prodes repetido na fonte: {ano}")
    registros[ano] = int(total)

consolidados = {ano: valor for ano, valor in registros.items() if ano <= LIMITE}
if not consolidados:
    raise ValueError("nenhum ano consolidado depois do corte")

ultimo = max(consolidados)
if ultimo != LIMITE:
    raise ValueError(f"o último ano consolidado é {ultimo}, esperado {LIMITE}")

descartados = sorted(set(registros) - set(consolidados))
print(f"Períodos brutos: {len(registros)} ({min(registros)} a {max(registros)})")
print(f"Publicados: {len(consolidados)} ({min(consolidados)} a {ultimo})")
print(f"Descartados por período aberto: {descartados}")


Períodos brutos: 39 (1988 a 2026)
Publicados: 38 (1988 a 2025)
Descartados por período aberto: [2026]


## Escrita do JSON canônico

Os pontos são gravados com `data` no primeiro dia do ano e `valor` inteiro em km², em ordem ascendente. Os metadados são os do contrato: `numero` / `contagem` / `nivel` / `anual`.


In [4]:
pontos = [
    {"data": f"{ano:04d}-01-01", "valor": consolidados[ano]}
    for ano in sorted(consolidados)
]

serie = {
    "versao": 1,
    "id": "desmatamento-amazonia-legal-anual",
    "unidade": {"tipo": "numero", "simbolo": "km²", "casas": 0},
    "eixo": "contagem",
    "medida": "nivel",
    "granularidade": "anual",
    "fonte": (
        "INPE — PRODES/BiomasBR (TerraBrasilis), taxa consolidada da Amazônia Legal, "
        "soma dos 9 estados calculada pelo site"
    ),
    "urlFonte": URL_FONTE,
    "dados": pontos,
}

DESTINO.mkdir(parents=True, exist_ok=True)
with ARQUIVO.open("w", encoding="utf-8") as arquivo:
    json.dump(serie, arquivo, ensure_ascii=False, indent=2)

print(f"{len(pontos)} pontos escritos em {relativo(ARQUIVO)}")
print(f"Primeiro ponto: {pontos[0]}")
print(f"Último ponto:   {pontos[-1]}")


38 pontos escritos em indicadores/meio-ambiente/desmatamento-amazonia-legal-anual.json
Primeiro ponto: {'data': '1988-01-01', 'valor': 21050}
Último ponto:   {'data': '2025-01-01', 'valor': 5731}


## Validação inline

A célula final falha se o contrato, o C15, a ordem, a unicidade dos anos, a finitude dos valores, a contagem ou a igualdade com a fonte forem violados. Também confere os valores citados em `docs/metodologia.md` e a ausência do ano do arquivo.


In [5]:
CHAVES = {
    "versao", "id", "unidade", "eixo", "medida", "granularidade", "fonte", "urlFonte", "dados",
}
assert set(serie) == CHAVES, sorted(set(serie) ^ CHAVES)
assert serie["versao"] == 1, serie["versao"]
assert serie["id"] == "desmatamento-amazonia-legal-anual", serie["id"]
assert serie["unidade"] == {"tipo": "numero", "simbolo": "km²", "casas": 0}, serie["unidade"]
assert serie["eixo"] == "contagem", serie["eixo"]
assert serie["medida"] == "nivel", serie["medida"]
assert serie["granularidade"] == "anual", serie["granularidade"]
assert serie["fonte"] == (
    "INPE — PRODES/BiomasBR (TerraBrasilis), taxa consolidada da Amazônia Legal, "
    "soma dos 9 estados calculada pelo site"
), serie["fonte"]
assert serie["urlFonte"] == PAGINA, serie["urlFonte"]

dados = serie["dados"]
datas = [ponto["data"] for ponto in dados]
anos = [int(data[:4]) for data in datas]

assert dados, "série vazia"
assert len(dados) == LIMITE - 1988 + 1, len(dados)
assert all(re.fullmatch(r"\d{4}-01-01", data) for data in datas), "data fora do C15"
assert datas == sorted(datas), "datas fora de ordem"
assert len(anos) == len(set(anos)), "ano repetido"
assert datas[0] == "1988-01-01", datas[0]
assert datas[-1] == f"{LIMITE}-01-01", datas[-1]
assert ANO_ARQUIVO not in anos, "estimativa do ano do arquivo publicada"
assert max(anos) == LIMITE, max(anos)

for ponto in dados:
    assert set(ponto) == {"data", "valor"}, sorted(ponto)
    assert isinstance(ponto["valor"], int) and not isinstance(ponto["valor"], bool), ponto
    assert math.isfinite(ponto["valor"]), ponto

por_ano = {int(ponto["data"][:4]): ponto["valor"] for ponto in dados}
esperado = {
    "1988": 21050,
    "1993": 14896,
    "1994": 14896,
    "1995": 29059,
    "2004": 27772,
    "2012": 4571,
    "2021": 13038,
    "2023": 9064,
    "2024": 6518,
    "2025": 5731,
}
for ano, valor in esperado.items():
    assert por_ano[int(ano)] == valor, (ano, por_ano[int(ano)], valor)

assert por_ano[1993] == por_ano[1994] == 14896, "a fonte repete 14.896 em 1993 e 1994"
assert max(por_ano, key=por_ano.get) == 1995, "máximo histórico deveria ser 1995"
assert min(por_ano, key=por_ano.get) == 2012, "mínimo deveria ser 2012"

fonte_por_ano = {
    int(periodo["endDate"]["year"]): int(
        sum(area["area"] for feature in periodo["features"] for area in feature["areas"])
    )
    for periodo in bruto["periods"]
    if int(periodo["endDate"]["year"]) <= LIMITE
}
assert por_ano == fonte_por_ano, "valor publicado difere da soma da fonte"

with ARQUIVO.open(encoding="utf-8") as arquivo:
    salvo = json.load(arquivo)
assert salvo == serie, "arquivo gravado difere do objeto validado"

print(
    f"ok: {len(dados)} anos únicos, {datas[0]} a {datas[-1]}, "
    f"1988={por_ano[1988]} km², {LIMITE}={por_ano[LIMITE]} km², sem {ANO_ARQUIVO}"
)


ok: 38 anos únicos, 1988-01-01 a 2025-01-01, 1988=21050 km², 2025=5731 km², sem 2026
